# 16 -- Full Ranking Curves (Fig. 6)

Cumulative rank curve per score: for a threshold t on the x-axis (the top fraction of a
model's layers, by score-assigned rank), the y-axis is the fraction of the 6 PTQ
architecture x dataset combinations in which the true top-damage layer falls within that
top-t fraction according to the score. A curve that rises steeply toward the left is a score
that reliably places the true top-damage layer near the front of its ranking; a curve close
to the diagonal is no better than a random ranking. This complements Table 6 (which reports
single point comparisons at k*=1,3,6) and Fig. 1 (full per-layer scatter) with a summary that
is normalized once across architectures and directly comparable between the three scores.

-> Fig. 6 in the report, Sec. 5.6. Saved as `figures/fig_06_ranking_curves.pdf`.

Source CSV: `normalized_ranks_loss.csv`, column `rank_pct` (true top-damage layer's
score-assigned rank, normalized by the number of layers in that architecture).

In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

RANKS_CSV = f"{REPO}/results/review_response/csv/normalized_ranks_loss.csv"

PRED_LABEL = {"raw_trh": "$S_{raw}$", "dwsq": "$S_{pert}$", "trh_dwsq": "$S_{hawq}$"}
PRED_ORDER = ["raw_trh", "dwsq", "trh_dwsq"]
PRED_COLOR = {"raw_trh": "#4C72B0", "dwsq": "#DD8452", "trh_dwsq": "#55A868"}

ranks = pd.read_csv(RANKS_CSV)
ranks[["model", "dataset", "predictor", "rank_pct"]]

,model,dataset,predictor,rank_pct
0,cnn,CIFAR10,raw_trh,0.3333
1,cnn,CIFAR10,dwsq,1.0000
2,cnn,CIFAR10,trh_dwsq,0.6667
3,cnn,IMAGENET100,raw_trh,0.5000
4,cnn,IMAGENET100,dwsq,1.0000
5,cnn,IMAGENET100,trh_dwsq,1.0000
6,resnet18_no_weights,CIFAR10,raw_trh,0.0476
7,resnet18_no_weights,CIFAR10,dwsq,0.9524
8,resnet18_no_weights,CIFAR10,trh_dwsq,0.5714
9,resnet18_no_weights,IMAGENET100,raw_trh,0.0476


For each score, the 6 `rank_pct` values (one per PTQ architecture x dataset combination) are
the thresholds at which the cumulative-hit curve steps up by 1/6. Evaluate the step function
on a shared, fine-grained grid of thresholds so all three curves can be plotted together.

In [2]:
thresholds = np.linspace(0.0, 1.0, 500)

curves = {}
for predictor in PRED_ORDER:
    hit_thresholds = ranks.loc[ranks["predictor"] == predictor, "rank_pct"].to_numpy()
    n_combos = len(hit_thresholds)
    fraction_hit = np.array([(hit_thresholds <= t).sum() / n_combos for t in thresholds])
    curves[predictor] = fraction_hit

{p: curves[p][-1] for p in PRED_ORDER}

{'raw_trh': np.float64(1.0),
 'dwsq': np.float64(1.0),
 'trh_dwsq': np.float64(1.0)}

Plot the three cumulative curves against the no-information diagonal (a score that ranks
layers uniformly at random would, in expectation, follow y=t).

In [3]:
fig, ax = plt.subplots(figsize=(4.2, 3.4), layout="constrained")

ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1.0, label="Zufalls-Rang (Diagonale)")

for predictor in PRED_ORDER:
    ax.step(thresholds, curves[predictor], where="post", color=PRED_COLOR[predictor],
            linewidth=1.8, label=PRED_LABEL[predictor])

ax.set_xlabel("Top-Rang-Schwelle $t$ (Anteil der Schichten)", fontsize=8.5)
ax.set_ylabel("Anteil der 6 PTQ-Kombinationen\nmit Top-Damage-Layer in Top-$t$", fontsize=8.5)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.tick_params(labelsize=7.5)
ax.legend(fontsize=7.5, loc="lower right", frameon=False)

fig.savefig(f"{FIG_DIR}/fig_06_ranking_curves.pdf")
fig.savefig(f"{FIG_DIR}/fig_06_ranking_curves.png", dpi=200)
plt.close(fig)

## Output

- `figures/fig_06_ranking_curves.pdf` -- Fig. 6 of the report (embedded in `sections/05_results.tex`)
- `figures/fig_06_ranking_curves.png` -- PNG copy for quick preview